In [ ]:
from statistics import stdev
import os
import yaml
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme()

In [ ]:
paths = {
    'Mistral 7B v0.3': '../experiments/l1ra-2025-03-02_21:38:32/',
    'Llama 3.1 8B': '../experiments/l1ra-2025-03-03_06:50:10/'
}

In [ ]:
df = None
for model, path in paths.items():
    tmp = pd.read_csv(os.path.join(path, 'rank_evolution.csv'))

    if tmp['step'].min() > 0:
        tmp_ = tmp[tmp['step'] == tmp['step'].min()].copy()
        tmp_['step'] = 0
        tmp_[[c for c in tmp_.columns if c not in ['step', 'layer']]] = 16
        tmp = pd.concat([tmp_, tmp], axis=0)

    config_file = [elem for elem in os.listdir(path) if elem.endswith('.yml') and elem != 'report.yml'][0]
    with open(os.path.join(path, config_file), "r") as file:
        config = yaml.safe_load(file)

    tmp['step'] = tmp['step'] // config['training_args'].get('gradient_accumulation_steps', 1)
    tmp['layer'] = tmp['layer'] + 1

    tmp['model'] = model
    df = tmp if df is None else pd.concat([df, tmp], axis=0)
df

In [ ]:
df_evolution = pd.DataFrame([{'Adapter rank': row[c], 'Step': row['step'], 'Matrix': c, 'Layer': row['layer'], 'Model': row['model']} for _, row in df.iterrows() for c in df.columns if c not in ['step', 'layer', 'model']])
df_evolution

In [ ]:
for model in df['model'].unique():
    df_last = df[df['model'] == model]
    df_last = df_last[df_last['step'] == df_last['step'].max()].drop(columns=['step', 'model']).set_index("layer")
    fig = plt.figure(figsize=(16,4))
    sns.heatmap(df_last.T, cmap="crest", annot=True, fmt="g", cbar_kws={'label': 'Adapter rank'})
    plt.ylabel('Matrix')
    plt.xlabel('Layer')
    plt.show()

    model_name = model.lower().replace(' ', '_')
    fig.savefig(f'./{model_name}_final_rank.pdf', bbox_inches='tight')

In [ ]:
vmin = df.drop(columns=["step", "layer", 'model']).min().min()
vmax = df.drop(columns=["step", "layer", 'model']).max().max()

In [ ]:
for model in df['model'].unique():
    g = df[df['model'] == model]

    steps = g["step"].unique()

    model_name = model.lower().replace(' ', '_')

    for step in steps:
        tmp = g[g['step'] == step].drop(columns=['step', 'model']).set_index('layer')

        fig = plt.figure(figsize=(16,4))
        sns.heatmap(tmp.T, cmap="crest", annot=True, vmin=vmin, vmax=vmax, annot_kws={"fontsize":10}, cbar_kws={'label': 'Adapter rank'})
        plt.ylabel('Matrix')
        plt.xlabel('Layer')

        fig.savefig(f'./{model_name}_rank_evolution_step_{str(step).zfill(4)}.pdf', bbox_inches='tight')

        plt.close(fig)

In [ ]:
for model in df['model'].unique():
    g = df_evolution[df_evolution['Model'] == model]

    fig = plt.figure(figsize=(8,4))
    sns.lineplot(g, x='Step', y='Adapter rank', hue='Matrix', errorbar=None)  # 'sd')
    plt.legend(ncol=3, title='Matrix', loc='upper left')
    plt.ylim(vmin, vmax)

    plt.show()

    model_name = model.lower().replace(' ', '_')
    fig.savefig(f'./{model_name}_avg_rank_evolution.pdf', bbox_inches='tight')

In [ ]:
tmp = pd.concat([
    df_evolution[
        (df_evolution['Model'] == model) &
        (df_evolution['Step'] == df_evolution[df_evolution['Model'] == model]['Step'].max())
    ] for model in df['model'].unique()
], axis=0)
fig = plt.figure(figsize=(6,6))
sns.barplot(df_evolution, x='Matrix', y='Adapter rank', hue='Model', errorbar='sd')

plt.show()

fig.savefig(f'./matrixwise_avg_final_rank_comparison.pdf', bbox_inches='tight')

In [ ]:
tmp = pd.concat([
    df_evolution[
        (df_evolution['Model'] == model) &
        (df_evolution['Step'] == df_evolution[df_evolution['Model'] == model]['Step'].max())
    ] for model in df['model'].unique()
], axis=0)
fig = plt.figure(figsize=(8,4))
sns.lineplot(df_evolution, x='Layer', y='Adapter rank', hue='Model', errorbar=None)  # 'sd')

plt.show()

fig.savefig(f'./layerwise_avg_final_rank_comparison.pdf', bbox_inches='tight')